### RAG pipelines - Data Ingestion to Vector DB Pipeline

In [7]:
import os
from langchain_community.document_loaders import PyMuPDFLoader, PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path



/var/folders/23/gwm9fvns0qsd6t2thxlcfwz00000gn/T/ipykernel_4118/870015536.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyMuPDFLoader, PyPDFLoader


In [8]:
### Read all the pdf's inside the directory
def process_all_pdfs(pdf_directory):
    """Process all PDF files in a directory"""
    all_documents = []
    pdf_dir = Path(pdf_directory)

    # Find all PDF files recursively
    pdf_files = list(pdf_dir.glob("**/*.pdf"))

    print(f"Found {len(pdf_files)} PDF files to process")

    for pdf_file in pdf_files:
        print(f"\nProcessing: {pdf_file.name}")
        try:
            loader = PyPDFLoader(str(pdf_file))
            documents = loader.load()

            ## Add source information to metadata
            for doc in documents:
                doc.metadata['source_file'] = pdf_file.name
                doc.metadata['file_type'] = 'pdf'

            all_documents.extend(documents)
            print(f"Loaded {len(documents)} pages")

        except Exception as e:
            print(f"Error: {e}")

    print(f"\nTotal pages loaded: {len(all_documents)}")
    return all_documents

## Process all PDFs in the data directory
all_pdf_documents = process_all_pdfs("../data")


Ignoring wrong pointing object 8 0 (offset 0)


Found 3 PDF files to process

Processing: sample-pdf-1.pdf
Loaded 3 pages

Processing: sample-pdf-3.pdf
Loaded 5 pages

Processing: sample-pdf-2.pdf
Loaded 1 pages

Total pages loaded: 9


In [10]:
all_pdf_documents

[Document(metadata={'producer': 'Mac OS X 10.11.3 Quartz PDFContext', 'creator': 'Word', 'creationdate': "D:20160319061844Z00'00'", 'title': 'Sample PDF', 'author': 'Philip Hutchison', 'subject': '', 'moddate': "D:20160319061844Z00'00'", 'keywords': '', 'aapl:keywords': '[]', 'source': '../data/pdf/sample-pdf-1.pdf', 'total_pages': 3, 'page': 0, 'page_label': '1', 'source_file': 'sample-pdf-1.pdf', 'file_type': 'pdf'}, page_content='1\t\nSample PDF  Created for testing PDFObject  This PDF is three pages long. Three long pages. Or three short pages if you’re optimistic. Is it the same as saying “three long minutes”, knowing that all minutes are the same duration, and one cannot possibly be longer than the other? If these pages are all the same size, can one possibly be longer than the other?  I digress. Here’s some Latin. Lorem ipsum dolor sit amet, consectetur adipiscing elit. Integer nec odio. Praesent libero. Sed cursus ante dapibus diam. Sed nisi. Nulla quis sem at nibh elementum im

In [12]:
### Text splitting get into chunks

def split_documents(documents, chunk_size=1000, chunk_over_lap=200):
    """Split documents into smaller chhunks for better RAG performance"""
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap = chunk_over_lap,
        length_function = len,
        separators=["\n\n", "\n", " ", ""]
    )
    split_docs = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks")

    ## show example of a chunk
    if split_docs:
        print(F"\nExample chunk:")
        print(F"\nContent:, {split_docs[0].page_content[:200]}...")
        print(F"\nExample chunk: {split_docs[0].metadata}")

    return split_docs

In [13]:
chunks = split_documents(all_pdf_documents)
chunks

Split 9 documents into 34 chunks

Example chunk:

Content:, 1...

Example chunk: {'producer': 'Mac OS X 10.11.3 Quartz PDFContext', 'creator': 'Word', 'creationdate': "D:20160319061844Z00'00'", 'title': 'Sample PDF', 'author': 'Philip Hutchison', 'subject': '', 'moddate': "D:20160319061844Z00'00'", 'keywords': '', 'aapl:keywords': '[]', 'source': '../data/pdf/sample-pdf-1.pdf', 'total_pages': 3, 'page': 0, 'page_label': '1', 'source_file': 'sample-pdf-1.pdf', 'file_type': 'pdf'}


[Document(metadata={'producer': 'Mac OS X 10.11.3 Quartz PDFContext', 'creator': 'Word', 'creationdate': "D:20160319061844Z00'00'", 'title': 'Sample PDF', 'author': 'Philip Hutchison', 'subject': '', 'moddate': "D:20160319061844Z00'00'", 'keywords': '', 'aapl:keywords': '[]', 'source': '../data/pdf/sample-pdf-1.pdf', 'total_pages': 3, 'page': 0, 'page_label': '1', 'source_file': 'sample-pdf-1.pdf', 'file_type': 'pdf'}, page_content='1'),
 Document(metadata={'producer': 'Mac OS X 10.11.3 Quartz PDFContext', 'creator': 'Word', 'creationdate': "D:20160319061844Z00'00'", 'title': 'Sample PDF', 'author': 'Philip Hutchison', 'subject': '', 'moddate': "D:20160319061844Z00'00'", 'keywords': '', 'aapl:keywords': '[]', 'source': '../data/pdf/sample-pdf-1.pdf', 'total_pages': 3, 'page': 0, 'page_label': '1', 'source_file': 'sample-pdf-1.pdf', 'file_type': 'pdf'}, page_content='Sample PDF  Created for testing PDFObject  This PDF is three pages long. Three long pages. Or three short pages if you’re

### Embedding and vectorStoreDB

In [14]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity

In [15]:
class EmbeddingManager:
    """Handles document embedding generation using SentenceTransformer"""

    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        """
        Initialize the embedding manager
        
        Args:
            model_name: HuggingFace model name for sentence embeddings
        """
        self.model_name = model_name
        self.model = None
        self._load_model()

    def _load_model(self):
        """Load the SentenceTransformer model"""
        try:
            print(f"Loading embedding model: {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print(f"Model loaded successfully, Embedding dimensions: {self.model.get_embedding_dimension()}")
        except Exception as e:
            print(f"Error loading model {self.model.name}: {e}")

    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        """
        Generate embeddings for a list of texts
        Args:
            texts: List of text strings to embed 
        Returns:
            numpy array of embeddings with shape (len(texts), embedding_dim)
        """
        if not self.model:
            raise ValueError("Model not found")

        print(f"Generating embeddings for {len(texts)} texts...")
        embeddings = self.model.encode(texts, show_progress_bar=True)
        print(f"Generated embeddings with shape: {embeddings.shape}")
        return embeddings

## initialize the embedding manager
embedding_manager = EmbeddingManager()
embedding_manager

Loading embedding model: all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 10343.41it/s]


Model loaded successfully, Embedding dimensions: 384


### VectorStore

In [16]:
class VectorStore:
    """Manages document embeddings in a ChromaDB vector store"""

    def __init__(self, collection_name: str="pdf_documents", persist_directory: str = "../data/vector_store"):
        """
        Initialize the vector store
        
        Args:
            collection_name: Name of the ChromaDB collection
            persist_directory: Directory to persist the vector store
        """
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store()

    def _initialize_store(self):
        """Initialize ChromaDB client and collection"""
        try:
            ## create persistent chromaDB client
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persist_directory)

            ## get or create collection
            self.collection = self.client.get_or_create_collection(
                name = self.collection_name,
                metadata={"description": "PDF document embeddings for RAG"}
            )
            print(f"Vector store initialized. Collection: {self.collection_name}")
            print(f"Existing documents in Collection: {self.collection.count()}")
            
        except Exception as e:
            print(f"Error initializing vector store: {e}")
            raise

    def add_documents(self, documents: List[Any], embeddings: np.ndarray):
        """
        Add documents and their embeddings to the vector store
        
        Args:
            documents: List of LangChain documents
            embeddings: Corresponding embeddings for the documents
        """
        if len(documents) != len(embeddings):
            raise ValueError("Number of documents must match number of embeddings")

        print(f"Adding {len(documents)} documents to vector store...")

        ## prepare data for chromaDB
        ids = []
        metadatas = []
        documents_text = []
        embeddings_list = []

        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            # Generate unique ID
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)
            
            # Prepare metadata
            metadata = dict(doc.metadata)
            metadata['doc_index'] = i
            metadata['content_length'] = len(doc.page_content)
            metadatas.append(metadata)
            
            # Document content
            documents_text.append(doc.page_content)
            
            # Embedding
            embeddings_list.append(embedding.tolist())
        
        # Add to collection
        try:
            self.collection.add(
                ids=ids,
                embeddings=embeddings_list,
                metadatas=metadatas,
                documents=documents_text
            )
            print(f"Successfully added {len(documents)} documents to vector store")
            print(f"Total documents in collection: {self.collection.count()}")
            
        except Exception as e:
            print(f"Error adding documents to vector store: {e}")
            raise

vectorstore=VectorStore()
vectorstore


Vector store initialized. Collection: pdf_documents
Existing documents in Collection: 34


In [17]:
## Generate embeddings of texts
texts = [doc.page_content for doc in chunks]
embeddings = embedding_manager.generate_embeddings(texts)
##store in Vector database
vectorstore.add_documents(chunks,embeddings)

Generating embeddings for 34 texts...


Batches: 100%|██████████| 2/2 [00:00<00:00,  3.90it/s]

Generated embeddings with shape: (34, 384)
Adding 34 documents to vector store...
Successfully added 34 documents to vector store
Total documents in collection: 68


### Retriever Pipeline From VectorStore

In [19]:
class RAGRetriever:
    """Handles query-based retrieval from the vector store"""
    
    def __init__(self, vector_store: VectorStore, embedding_manager: EmbeddingManager):
        """
        Initialize the retriever
        
        Args:
            vector_store: Vector store containing document embeddings
            embedding_manager: Manager for generating query embeddings
        """
        self.vector_store = vector_store
        self.embedding_manager = embedding_manager

    def retrieve(self, query: str, top_k: int = 5, score_threshold: float = 0.0) -> List[Dict[str, Any]]:
        """
        Retrieve relevant documents for a query
        
        Args:
            query: The search query
            top_k: Number of top results to return
            score_threshold: Minimum similarity score threshold
            
        Returns:
            List of dictionaries containing retrieved documents and metadata
        """
        print(f"Retrieving documents for query: '{query}'")
        print(f"Top K: {top_k}, Score threshold: {score_threshold}")
        
        # Generate query embedding
        query_embedding = self.embedding_manager.generate_embeddings([query])[0]
        
        # Search in vector store
        try:
            results = self.vector_store.collection.query(
                query_embeddings=[query_embedding.tolist()],
                n_results=top_k
            )
            
            # Process results
            retrieved_docs = []
            
            if results['documents'] and results['documents'][0]:
                documents = results['documents'][0]
                metadatas = results['metadatas'][0]
                distances = results['distances'][0]
                ids = results['ids'][0]
                
                for i, (doc_id, document, metadata, distance) in enumerate(zip(ids, documents, metadatas, distances)):
                    # Convert distance to similarity score (ChromaDB uses cosine distance)
                    similarity_score = 1 - distance
                    
                    if similarity_score >= score_threshold:
                        retrieved_docs.append({
                            'id': doc_id,
                            'content': document,
                            'metadata': metadata,
                            'similarity_score': similarity_score,
                            'distance': distance,
                            'rank': i + 1
                        })
                
                print(f"Retrieved {len(retrieved_docs)} documents (after filtering)")
            else:
                print("No documents found")
            
            return retrieved_docs
            
        except Exception as e:
            print(f"Error during retrieval: {e}")
            return []

rag_retriever=RAGRetriever(vectorstore,embedding_manager)

rag_retriever

In [20]:
rag_retriever.retrieve("VRML Mission Statement")

Retrieving documents for query: 'VRML Mission Statement'
Top K: 5, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 18.64it/s]

Generated embeddings with shape: (1, 384)
Retrieved 5 documents (after filtering)


[{'id': 'doc_02f767d9_26',
  'content': 'Virtual Reality Markup Language (VRML) was coined, and the group resolved to begin\nspecification work after the conference. The word ’Markup’ was later changed to\n’Modeling’ to reflect the graphical nature of VRML.',
  'metadata': {'creator': 'Pdf995',
   'producer': 'GNU Ghostscript 7.05',
   'source': '../data/pdf/sample-pdf-3.pdf',
   'page_label': '3',
   'source_file': 'sample-pdf-3.pdf',
   'title': 'PDF',
   'creationdate': '12/12/2003 17:30:12',
   'total_pages': 5,
   'doc_index': 26,
   'file_type': 'pdf',
   'keywords': 'pdf, create pdf, software, acrobat, adobe',
   'page': 2,
   'content_length': 214,
   'subject': 'Create PDF with Pdf 995',
   'author': 'Software 995'},
  'similarity_score': 0.1267470121383667,
  'distance': 0.8732529878616333,
  'rank': 1},
 {'id': 'doc_99572359_26',
  'content': 'Virtual Reality Markup Language (VRML) was coined, and the group resolved to begin\nspecification work after the conference. The word

### Integration Vectordb Context pipeline with LLM output

In [27]:
## simple RAG pipeline with GROQ LLM
from langchain_groq import ChatGroq
import os
from dotenv import load_dotenv
load_dotenv()

## Initialize the GROQ LLM
groq_api_key = os.getenv("GROQ_API_KEY")
if not groq_api_key:
    raise ValueError("GROQ_API_KEY not found in environment variables")

llm = ChatGroq(api_key=groq_api_key, model="llama-3.3-70b-versatile", temperature=0.1, max_tokens=1024)

## simple RAG function: retricve context + generate response
def rag_simple(query, retriever, llm, top_k=3):
    ## retrive the context
    results = retriever.retrieve(query, top_k=top_k)
    context = "\n\n".join([doc['content'] for doc in results]) if results else "No relevant context found."

    ## generate the answers using GROQ LLM
    prompt = f"""Answer the following question based on the context provided.
    Context:{context}
    Question: {query}
    Answer:"""

    response = llm.invoke([prompt.format(context=context, query=query)])
    return response.content


In [28]:
answer = rag_simple("VRML Mission Statement", rag_retriever, llm)
answer

Retrieving documents for query: 'VRML Mission Statement'
Top K: 3, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 28.75it/s]

Generated embeddings with shape: (1, 384)
Retrieved 3 documents (after filtering)


"There is no explicit VRML mission statement provided in the given context. The context primarily discusses the origins and initial development of Virtual Reality Markup Language (VRML), including the change of 'Markup' to 'Modeling', the creation of a mailing list for specification development, and the process of setting requirements and searching for adaptable technologies. It does not outline a specific mission statement for VRML."

### Enhanced RAG Pipeline Features

In [33]:
# --- Enhanced RAG Pipeline Features ---
def rag_advanced(query, retriever, llm, top_k=5, min_score=0.2, return_context=False):
    """
    RAG pipeline with extra features:
    - Returns answer, sources, confidence score, and optionally full context.
    """
    results = retriever.retrieve(query, top_k=top_k, score_threshold=min_score)
    if not results:
        return {'answer': 'No relevant context found.', 'sources': [], 'confidence': 0.0, 'context': ''}
    
    # Prepare context and sources
    context = "\n\n".join([doc['content'] for doc in results])
    sources = [{
        'source': doc['metadata'].get('source_file', doc['metadata'].get('source', 'unknown')),
        'page': doc['metadata'].get('page', 'unknown'),
        'score': doc['similarity_score'],
        'preview': doc['content'][:300] + '...'
    } for doc in results]
    confidence = max([doc['similarity_score'] for doc in results])
    
    # Generate answer
    prompt = f"""Use the following context to answer the question concisely.\nContext:\n{context}\n\nQuestion: {query}\n\nAnswer:"""
    response = llm.invoke([prompt.format(context=context, query=query)])
    
    output = {
        'answer': response.content,
        'sources': sources,
        'confidence': confidence
    }
    if return_context:
        output['context'] = context
    return output

# Example usage:
result = rag_advanced("Cold you please tell me about VRML Mission Statement", rag_retriever, llm, top_k=3, min_score=0.1, return_context=True)
print("Answer:", result['answer'])
print("Sources:", result['sources'])
print("Confidence:", result['confidence'])
print("Context Preview:", result['context'][:300])

Retrieving documents for query: 'Cold you please tell me about VRML Mission Statement'
Top K: 3, Score threshold: 0.1
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 35.62it/s]

Generated embeddings with shape: (1, 384)
Retrieved 2 documents (after filtering)


Answer: There is no mention of a VRML mission statement in the provided context.
Sources: [{'source': 'sample-pdf-3.pdf', 'page': 3, 'score': 0.10286206007003784, 'preview': 'APPROVED\nShortly after the Geneva BOF session, the www-vrml mailing list was created to discuss\nthe development of a specification for the first version of VRML. The response to the list\ninvitation was overwhelming: within a week, there were over a thousand members. After\nan initial settling-in peri...'}, {'source': 'sample-pdf-3.pdf', 'page': 3, 'score': 0.10286206007003784, 'preview': 'APPROVED\nShortly after the Geneva BOF session, the www-vrml mailing list was created to discuss\nthe development of a specification for the first version of VRML. The response to the list\ninvitation was overwhelming: within a week, there were over a thousand members. After\nan initial settling-in peri...'}]
Confidence: 0.10286206007003784
Context Preview: APPROVED
Shortly after the Geneva BOF session, the www-vrml mailing li

In [34]:
# --- Advanced RAG Pipeline: Streaming, Citations, History, Summarization ---
from typing import List, Dict, Any
import time

class AdvancedRAGPipeline:
    def __init__(self, retriever, llm):
        self.retriever = retriever
        self.llm = llm
        self.history = []  # Store query history

    def query(self, question: str, top_k: int = 5, min_score: float = 0.2, stream: bool = False, summarize: bool = False) -> Dict[str, Any]:
        # Retrieve relevant documents
        results = self.retriever.retrieve(question, top_k=top_k, score_threshold=min_score)
        if not results:
            answer = "No relevant context found."
            sources = []
            context = ""
        else:
            context = "\n\n".join([doc['content'] for doc in results])
            sources = [{
                'source': doc['metadata'].get('source_file', doc['metadata'].get('source', 'unknown')),
                'page': doc['metadata'].get('page', 'unknown'),
                'score': doc['similarity_score'],
                'preview': doc['content'][:120] + '...'
            } for doc in results]
            # Streaming answer simulation
            prompt = f"""Use the following context to answer the question concisely.\nContext:\n{context}\n\nQuestion: {question}\n\nAnswer:"""
            if stream:
                print("Streaming answer:")
                for i in range(0, len(prompt), 80):
                    print(prompt[i:i+80], end='', flush=True)
                    time.sleep(0.05)
                print()
            response = self.llm.invoke([prompt.format(context=context, question=question)])
            answer = response.content

        # Add citations to answer
        citations = [f"[{i+1}] {src['source']} (page {src['page']})" for i, src in enumerate(sources)]
        answer_with_citations = answer + "\n\nCitations:\n" + "\n".join(citations) if citations else answer

        # Optionally summarize answer
        summary = None
        if summarize and answer:
            summary_prompt = f"Summarize the following answer in 2 sentences:\n{answer}"
            summary_resp = self.llm.invoke([summary_prompt])
            summary = summary_resp.content

        # Store query history
        self.history.append({
            'question': question,
            'answer': answer,
            'sources': sources,
            'summary': summary
        })

        return {
            'question': question,
            'answer': answer_with_citations,
            'sources': sources,
            'summary': summary,
            'history': self.history
        }

# Example usage:
adv_rag = AdvancedRAGPipeline(rag_retriever, llm)
result = adv_rag.query("Cold you please tell me about VRML Mission Statement", top_k=3, min_score=0.1, stream=True, summarize=True)
print("\nFinal Answer:", result['answer'])
print("Summary:", result['summary'])
print("History:", result['history'][-1])

Retrieving documents for query: 'Cold you please tell me about VRML Mission Statement'
Top K: 3, Score threshold: 0.1
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 36.39it/s]

Generated embeddings with shape: (1, 384)
Retrieved 2 documents (after filtering)
Streaming answer:
Use the following context to answer the question concisely.
Context:
APPROVED
Shortly after the Geneva BOF session, the www-vrml mailing list was created to discuss
the development of a specification for the first version of VRML. The response to the list
invitation was overwhelming: within a week, there were over a th

ousand members. After
an initial settling-in period, list moderator Mark Pesce of Labyrinth Group announced his
intention to have a draft version of the specification ready by the WWW Fall 1994
conference, a mere five months away. There was general agreement on the list that, while
this schedule was aggressive, it was achievable provided that the requirements for the
first version were not too ambitious and that VRML could be adapted from an existing
solution. The list quickly agreed upon a set of requirements for the first version, and
began a search for technologies which could be adapted to fit the needs of VRML.
The search for existing technologies turned up a several worthwhile candidates. After

APPROVED
Shortly after the Geneva BOF session, the www-vrml mailing list was created to discuss
the development of a specification for the first version of VRML. The response to the list
invitation was overwhelming: within a week, there were over a thousand members. After
an initial settl